[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-01-fundamentals.ipynb)

---
# Day 1 · dlt Fundamentals and Architecture
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Foundation

> **Goal for today:** Understand what dlt is, how its core abstractions fit together, and get your first pipeline running against a local DuckDB database.

---
## What is dlt?

**dlt (data load tool)** is an open-source Python library that turns any Python function into a production-grade data pipeline. It handles the boring-but-critical stuff for you:

| What you'd normally build | What dlt gives you for free |
|---|---|
| Schema inference | Automatic, with evolution support |
| Type coercion | Built-in, configurable |
| Deduplication / upserts | `merge` write disposition |
| Incremental state | Persisted in the destination |
| Secrets management | `.dlt/secrets.toml` or env vars |
| Normalisation of nested JSON | Automatic table splitting |

The philosophy: **dlt is opinionated so you don't have to be.** Don't fight the defaults until you understand them.


---
## Core architecture: 4 abstractions

Every dlt pipeline follows the same flow:

```
Source  ──▶  Resource  ──▶  Pipeline  ──▶  Destination
```

| Abstraction | What it is | Analogy |
|---|---|---|
| **Source** | A logical group of related resources (e.g. a whole API) | A database |
| **Resource** | A single stream of data (e.g. one endpoint) | A table |
| **Pipeline** | Orchestrates the run: schema, state, credentials | The ETL job |
| **Destination** | Where data lands: DuckDB, BigQuery, Snowflake, etc. | The warehouse |

You define Sources and Resources. dlt owns the Pipeline and Destination wiring.


---
## Step 1 · Install dlt

We'll use DuckDB as our destination — it requires no credentials and runs fully in-process. Perfect for learning.

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · Hello World pipeline

The simplest possible pipeline: a Python list as the source, DuckDB as the destination.

Read the code carefully before running — every line is doing real work.

In [ ]:
import dlt

# 1. Define a resource — the simplest form is a plain Python generator
@dlt.resource
def users():
    yield [
        {"id": 1, "name": "Alice",   "role": "engineer"},
        {"id": 2, "name": "Bob",     "role": "analyst"},
        {"id": 3, "name": "Charlie", "role": "engineer"},
    ]

# 2. Create a pipeline — give it a name, a destination, and a dataset name
pipeline = dlt.pipeline(
    pipeline_name="hello_world",
    destination="duckdb",
    dataset_name="raw",
)

# 3. Run it
info = pipeline.run(users())
print(info)

**What just happened?**
- dlt inferred the schema from your dicts (column names + types)
- Created a `raw.users` table in a local `hello_world.duckdb` file
- Loaded 3 rows with `append` write disposition (default)

Let's verify by querying the result directly:

In [ ]:
# dlt exposes a SQL client for quick inspection
with pipeline.sql_client() as client:
    with client.execute_query("SELECT * FROM raw.users") as cursor:
        print(cursor.df())

Notice dlt added extra columns: `_dlt_id` (row hash) and `_dlt_load_id` (load run identifier). These are dlt's internal tracking columns — you'll see them in every table.

---
## Step 3 · Inspect the inferred schema

dlt built a schema automatically. Let's look at what it produced:

In [ ]:
schema = pipeline.default_schema
print(schema.to_pretty_yaml())

You'll see column names, inferred data types (`bigint`, `text`), and the dlt metadata columns. In production you'd pin these types explicitly — but auto-inference is fine for exploration.

---
## Step 4 · What happens on a second run?

Run the pipeline again with the **same data** and see what happens:

In [ ]:
info2 = pipeline.run(users())
print(info2)

with pipeline.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) as total FROM raw.users") as cursor:
        print(cursor.df())

**You now have 6 rows.** The default `append` disposition adds rows on every run — no deduplication. This is intentional and correct for append-only sources (logs, events).

For sources where you want upserts, you'd use `write_disposition="merge"` with a primary key — that's Day 3.

---
## Step 5 · A real-world shaped source

Real pipelines use `@dlt.source` to group related resources. Here's the pattern:

In [ ]:
@dlt.source
def my_app_db():
    """Simulates pulling from a multi-table application database."""

    @dlt.resource(primary_key="id", write_disposition="replace")
    def products():
        yield [
            {"id": 1, "name": "Widget A", "price": 9.99,  "category": "widgets"},
            {"id": 2, "name": "Widget B", "price": 14.99, "category": "widgets"},
            {"id": 3, "name": "Gadget X", "price": 49.99, "category": "gadgets"},
        ]

    @dlt.resource(primary_key="id", write_disposition="replace")
    def orders():
        yield [
            {"id": 101, "product_id": 1, "qty": 2, "status": "shipped"},
            {"id": 102, "product_id": 3, "qty": 1, "status": "pending"},
        ]

    return products, orders


pipeline2 = dlt.pipeline(
    pipeline_name="app_db",
    destination="duckdb",
    dataset_name="app",
)

info3 = pipeline2.run(my_app_db())
print(info3)

In [ ]:
# Inspect the loaded tables
with pipeline2.sql_client() as client:
    for table in ["app.products", "app.orders"]:
        print(f"\n── {table} ──")
        with client.execute_query(f"SELECT * FROM {table}") as cur:
            print(cur.df())

Notice `write_disposition="replace"` — the entire table is replaced on each run. Run `pipeline2.run(my_app_db())` again and the row count stays the same.

---
## Challenge — do this yourself

Modify the `users` resource to:
1. Add a `department` field to each user
2. Use `write_disposition="replace"` instead of the default `append`
3. Run the pipeline and verify there are still only 3 rows (not 6+)

In [ ]:
# Your solution here

@dlt.resource(write_disposition="replace")
def users_v2():
    yield ...  # fill this in

pipeline3 = dlt.pipeline(
    pipeline_name="challenge",
    destination="duckdb",
    dataset_name="raw",
)

info4 = pipeline3.run(users_v2())
print(info4)

<details>
<summary>Show solution</summary>

```python
@dlt.resource(write_disposition="replace")
def users_v2():
    yield [
        {"id": 1, "name": "Alice",   "role": "engineer", "department": "platform"},
        {"id": 2, "name": "Bob",     "role": "analyst",  "department": "analytics"},
        {"id": 3, "name": "Charlie", "role": "engineer", "department": "platform"},
    ]
```

The schema automatically picks up the new `department` column on first run.
</details>

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `@dlt.resource` | Wraps any generator into a loadable stream |
| `@dlt.source` | Groups multiple resources under one logical unit |
| `dlt.pipeline()` | Owns schema, state, credentials, and the run itself |
| `write_disposition` | `append` (default), `replace`, or `merge` |
| Schema inference | Automatic — but you can and should pin types in production |
| `_dlt_id` / `_dlt_load_id` | dlt's internal tracking columns — always present |

> **Tip:** dlt is opinionated — it handles schema inference, type coercion, and deduplication for you. Don't fight the defaults until you've shipped something.

---
## What's next

**Day 2** → `@dlt.source` and `@dlt.resource` in depth, the `rest_api` helper, pagination, and auth.

Mark Day 1 complete in your [tracker](../index.html).